In [ ]:
#imports

In [ ]:


model_detector_dict = get_model_detector_dict()

Datasets = ["annthyroid"] #1 seul pour l'exemple
model_combinaisons = [PReNet]

for dataset in Datasets:
    data = load_data(dataset)
    for models_combo in model_combinaisons:
        for t in range(trials):
            time_cost.start() # à écrire

            load_train_configs(dataset, models_combo)
            loaded_models = load_models()
            seed = set_seed(t)
            Collab = CoLearner(loaded_models, 
                               data, 
                               colearning_strategy,#None pour le moment car modèle fit tout seul pour le mvp
                               seed)
            results = Collab.train()
            results_analyse(results) # auc à obtenir pour le moment

            time_cost.stop() # à écrire
            results.log_time(time_cost())

results.save()
            



In [ ]:
#cotraining modèle 
for chapter in range(max_chapters):
    for model in models:
        model.fit() if chapter == 0 else model.train(epoch)
    
    # envoi pseudo
    for model in models:
        scores = model.predict_scores()
        data.update_pseudo_labels(model.name, scores, threshold=0.5)
    
    # décision commune ? ou étape par étape ? à implanter les 2
    ensemble_labels = data.get_ensemble_pseudo_labels("majority")
    
    auc = roc_auc_score(data.y_test, ensemble_scores)